In [14]:
# Imports
import os
import cv2
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow.keras.backend as K


In [15]:
# Constants
BASE_DIR = "./content/Training"
TUMOR_DIR = os.path.join(BASE_DIR, "Tumor")
STROKE_DIR = os.path.join(BASE_DIR, "Stroke")
ALL_IMAGES_DIR = os.path.join(BASE_DIR, "AllImages")
CSV_PATH = "labels.csv"

resize_width, resize_height = 224, 224

In [16]:
# Create the "AllImages" directory if it doesn't exist
os.makedirs(ALL_IMAGES_DIR, exist_ok=True)

# Initialize lists to store filenames and labels
filenames = []
tumor_labels = []
stroke_labels = []

In [17]:
def process_images(sub_dir, label_column):
    for class_label in ['yes', 'no']:
        class_path = os.path.join(sub_dir, class_label)
        if not os.path.exists(class_path):
            continue
        for img_name in os.listdir(class_path):
            src = os.path.join(class_path, img_name)
            dst = os.path.join(ALL_IMAGES_DIR, img_name)
            if not os.path.exists(dst):
                os.rename(src, dst)
            filenames.append(img_name)
            if label_column == 'tumor':
                tumor_labels.append(1 if class_label == 'yes' else 0)
                stroke_labels.append(0)
            elif label_column == 'stroke':
                stroke_labels.append(1 if class_label == 'yes' else 0)
                tumor_labels.append(0)

process_images(TUMOR_DIR, 'tumor')
process_images(STROKE_DIR, 'stroke')

In [18]:
# Create a DataFrame with the filenames and labels
df = pd.DataFrame({'filename': filenames, 'tumor': tumor_labels, 'stroke': stroke_labels})
df = df.drop_duplicates(subset=['filename'])
df.to_csv(CSV_PATH, index=False)

In [19]:
# Load labels and split into training and validation sets
df = pd.read_csv(CSV_PATH)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train DF: ", train_df)
print("Val DF: ", val_df)

Train DF:            filename  tumor  stroke
261    58 (19).jpg      0       1
809    83 (18).jpg      0       1
1405  106 (24).jpg      0       0
1786  119 (23).jpg      0       0
727    80 (30).jpg      0       1
...            ...    ...     ...
1638  114 (26).jpg      0       0
1095   92 (34).jpg      0       1
1130   93 (35).jpg      0       1
1294   102 (9).jpg      0       0
860    84 (39).jpg      0       1

[2203 rows x 3 columns]
Val DF:            filename  tumor  stroke
2740   99 (23).jpg      0       0
1911  123 (29).jpg      0       0
1263   101 (9).jpg      0       0
1456  108 (17).jpg      0       0
2254    52 (3).jpg      0       0
...            ...    ...     ...
315     66 (9).jpg      0       1
798     82 (8).jpg      0       1
759    81 (35).jpg      0       1
568    75 (14).jpg      0       1
644    77 (38).jpg      0       1

[551 rows x 3 columns]


In [20]:
# Data Generators
train_data_gen = ImageDataGenerator(preprocessing_function=preprocess_input, 
                                    horizontal_flip=True, vertical_flip=True, 
                                    rotation_range=30, zoom_range=0.3, shear_range=0.2)
val_data_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_data_gen.flow_from_dataframe(
    train_df,
    ALL_IMAGES_DIR,
    x_col='filename',
    y_col=['tumor', 'stroke'],
    target_size=(resize_width, resize_height),
    class_mode=None,
    batch_size=32,
    shuffle=True
)

validation_generator = val_data_gen.flow_from_dataframe(
    val_df,
    ALL_IMAGES_DIR,
    x_col='filename',
    y_col=['tumor', 'stroke'],
    target_size=(resize_width, resize_height),
    class_mode=None,
    batch_size=32,
    shuffle=False
)

train_labels = train_df[['tumor', 'stroke']].values
val_labels = val_df[['tumor', 'stroke']].values

# Class mapping is now not necessary as labels are separate
# But you can still print some info
print("Training samples:", train_generator.samples)
print("Validation samples:", validation_generator.samples)

print("Train labels shape:", train_labels.shape)
print("Validation labels shape:", val_labels.shape)

Found 2203 validated image filenames.
Found 551 validated image filenames.
Training samples: 2203
Validation samples: 551
Train labels shape: (2203, 2)
Validation labels shape: (551, 2)


In [21]:
def custom_data_generator(image_generator, labels):
    while True:
        images = next(image_generator)
        batch_size = images.shape[0]
        yield images, {'tumor_output': labels[:batch_size, 0], 'stroke_output': labels[:batch_size, 1]}

train_gen = custom_data_generator(train_generator, train_labels)
val_gen = custom_data_generator(validation_generator, val_labels)

In [22]:
def weighted_binary_crossentropy(weights):
    def loss(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        loss = -(
            weights[1] * y_true * K.log(y_pred) +
            weights[0] * (1 - y_true) * K.log(1 - y_pred)
        )
        return K.mean(loss)
    return loss

In [23]:
# Calculate weights for tumor and stroke
tumor_total = 155 + 98
tumor_weights = {0: tumor_total / (2 * 98), 1: tumor_total / (2 * 155)}  # For tumor: 0 (no), 1 (yes)

stroke_total = 950 + 1551
stroke_weights = {0: stroke_total / (2 * 1551), 1: stroke_total / (2 * 950)}  # For stroke: 0 (no), 1 (yes)

print("Tumor weights:", tumor_weights)
print("Stroke weights:", stroke_weights)

Tumor weights: {0: 1.2908163265306123, 1: 0.8161290322580645}
Stroke weights: {0: 0.806254029658285, 1: 1.316315789473684}


In [24]:
# VGG16 model
# Build the Model
vgg = VGG16(weights="imagenet", include_top=False, input_shape=(resize_width, resize_height, 3))

# Freeze layers and add custom layers
for layer in vgg.layers[:-8]:
    layer.trainable = False

In [25]:
def structure_model(bottom_model):
    top_model = bottom_model.output
    top_model = GlobalAveragePooling2D()(top_model)
    top_model = Dense(1024, activation='relu', kernel_regularizer=l2(0.01))(top_model)
    top_model = Dropout(0.3)(top_model)
    top_model = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(top_model)
    top_model = Dropout(0.3)(top_model)
    return top_model

model_head = structure_model(vgg)
tumor_output = Dense(1, activation='sigmoid', name='tumor_output')(model_head)
stroke_output = Dense(1, activation='sigmoid', name='stroke_output')(model_head)

In [26]:
model = Model(inputs=vgg.input, outputs=[tumor_output, stroke_output])

# Use custom weighted loss for tumor and stroke outputs
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss={
        'tumor_output': weighted_binary_crossentropy(tumor_weights),
        'stroke_output': weighted_binary_crossentropy(stroke_weights),
    },
    metrics={'tumor_output': 'accuracy', 'stroke_output': 'accuracy'}
)


In [27]:
# Callbacks
checkpoint = ModelCheckpoint('model/best_model.keras', save_best_only=True, monitor='val_loss')
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

# Train the Model
history = model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=int(np.ceil(train_generator.samples / train_generator.batch_size)),
    validation_steps=int(np.ceil(validation_generator.samples / validation_generator.batch_size)),
    epochs=30,
    callbacks=[checkpoint, lr_scheduler, early_stopping]
)

Epoch 1/30


TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [ ]:
# Evaluate the Model
val_predictions = model.predict(validation_generator)
predicted_tumor = (val_predictions[0] > 0.5).astype(int).reshape(-1)
predicted_stroke = (val_predictions[1] > 0.5).astype(int).reshape(-1)

true_tumor = val_df['tumor'].values
true_stroke = val_df['stroke'].values

print("Tumor Classification Report:")
print(classification_report(true_tumor, predicted_tumor))

print("Stroke Classification Report:")
print(classification_report(true_stroke, predicted_stroke))

In [ ]:
# Confusion Matrices
sns.heatmap(confusion_matrix(true_tumor, predicted_tumor), annot=True, fmt='d', cmap='Blues')
plt.title('Tumor Detection Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

sns.heatmap(confusion_matrix(true_stroke, predicted_stroke), annot=True, fmt='d', cmap='Greens')
plt.title('Stroke Detection Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [15]:
# Predict on a Sample Image
img_path = "./content/Training/Stroke/no/58 (1).jpg"
img = cv2.imread(img_path)
img = cv2.resize(img, (resize_width, resize_height), cv2.INTER_LANCZOS4)
img_array = preprocess_input(np.expand_dims(img, axis=0))

predictions = model.predict(img_array)
tumor_pred = 'Yes' if predictions[0] > 0.5 else 'No'
stroke_pred = 'Yes' if predictions[1] > 0.5 else 'No'

print(f"Predicted Tumor: {tumor_pred}")
print(f"Predicted Stroke: {stroke_pred}")